In [ ]:
# import 

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import pyplot
import os
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
import sys
from utils.utilities import find_best_grid_point, get_station_coords,form_xdate, get_anomalies
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import datetime as dt

from plotting import tol_colors # color schemes from https://personal.sron.nl/~pault/
from utils import process_data

#activate interactive figures
%matplotlib widget
#activate autoreload
%load_ext autoreload

## Add parent directory to syspath
parent_dir = os.path.abspath(os.path.join(os.path.dirname('.'), '..'))
if not parent_dir in sys.path:
    sys.path.append(parent_dir)

#save figures in...
dir_save = './output/check_data/'

In [ ]:
## Read all data
%autoreload 2
from input.read_wdc_data import AvailableData, create_data_reader

# File path
data_path = "../data/"

# if New data is added to ./data folder, adapt the dictionary in AvailableData
all_data = list(AvailableData)
print(all_data)

#####---------- TO ADAPT ---------------#####
selected_data = ['CO2', 'CO2_flask', 
                 'CO', 'CO_flask', 
                 'CH4', 'CH4_flask', 
                 'O3'
                 ] # define data to read in. If empty, all data is used 
## 
processing_kwargs = { 
    'FLASK_FLAG_CORR' : True # exclude flagged flask-data
}
#####-----------------------------------#####

datasets = [] # initialize list of all datasets 
# read in data
for sel in (selected_data if selected_data else all_data):
    #define where the data has to be read from
    data_reader =  create_data_reader(data_path=data_path,dataset=sel,**processing_kwargs) #creates an instance of the desired data_reader class
    print(f"Data from {data_reader.__class__.__name__} for {sel}:")

    # call the data-reading function on that instance: 
    data = data_reader.read_data() 
    # call the data-processing
    data = data_reader.process_data(data)

    # prepare merged dataset
    data = data.drop(columns='endtime') # problem when merging datasets (because of NaT?), so better remove endtime
    ds = data.to_xarray()
    ds = ds.assign_coords(dataset=sel)
    ds['species'] = data_reader.species
    ds['unit']  = np.unique(ds.unit.dropna(dim='time'))[0]
    datasets.append(ds)

# save all in one xarray dataset
ds_all = xr.concat(datasets,dim="dataset")


In [ ]:
ds_all

In [ ]:
## check flask data
sel_spec = "CO2_flask"

plt.figure()
ds = ds_all.sel(dataset=sel_spec)
ds = ds.where(~np.isnan(ds.value), drop=True)  # remove times where we have no data
ds.QCflag.plot(ls="", marker=".")
plt.show()

In [ ]:
# select all flask species and plot flags: (set 'FLASK_FLAG_CORR' : False)
sel_species = [s for s in selected_data if "flask" in s]

fig, axs = plt.subplots(len(sel_species), 1, sharex=True)
for s, ax in zip(sel_species, axs):
    ds = ds_all.sel(dataset=s)
    ds.value.plot(ax=ax, ls="", marker=".")

    # check QCflag = 3
    ds.where(ds.QCflag == 3).value.plot(ax=ax, ls="", marker=".", c="r")

    # check original flag (reject if first character is not '.' ) -> gives the same!
    # mask_flag = [str(s)[0] != '.' if isinstance(s, str) else False for s in ds.ORG_QCflag.values]
    # mask_dataarray = xr.DataArray(mask_flag, dims='time', coords={'time': ds['time']})
    # if any(mask_flag):
    #    ds.where(mask_dataarray,drop=True).value.plot(ax=ax,ls='',marker='x',c='g')

In [ ]:
### Plot all species
# species to plot (all available, or make a selection)
sel_species = ds_all.species
unique_species = np.unique(sel_species)

# moving window
mw = 24 * 10  # hours
# select time period
t1 = "2020-01-01"
t2 = "2023-12-31"


# function to plot each subplot
def plot_data(ds_temp, i_s, mw_temp, ax=None, **kwargs):
    if ax is None:
        ax = plt.gca()
    ds_temp.plot(ax=ax, ls="", marker=".", alpha=0.7, label=str(i_s))
    # moving mean
    if ("flask" in i_s) == False:  # no moving mean for flask
        ds_temp.rolling(time=mw_temp, center=True, min_periods=mw_temp / 2).mean().plot(
            ax=ax, ls="-", c="k", label=f"Moving Mean ({int(mw_temp/24)}days)"
        )


## Start figure
fig, axs = plt.subplots(
    len(unique_species),
    1,
    sharex=True,
    figsize=(8, len(unique_species) * 2),
    layout="constrained",
)

for s, ax in zip(unique_species, axs):
    ds_sel = ds_all.where(ds_all.species == s, drop=True).sel(time=slice(t1, t2))
    if len(ds_sel.dataset) > 1:  # several datasets with same species
        for ii in ds_sel.dataset:
            # plot data (adapt moving window for flask)
            plot_data(
                ds_sel.sel(dataset=ii).value,
                ii.item(),
                mw * 3 if ii.astype(str).str.contains("flask") else mw,
                ax,
            )
    else:
        plot_data(ds_sel.sel(dataset=s).value, s, mw, ax)

    ax.set_ylabel(f"{ds_sel.species.values[0]} ({ds_sel.unit.values[0]})")
    ax.set_xlabel("")
    ax.set_title("")
# ax.set_xlim(np.array([t1, t2], dtype="datetime64"))

handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper left")
fig.align_ylabels()

plt.suptitle("Mt. Kenya GAW station")

plt.savefig(
    f"{dir_save}timeseries_{t1[0:4]}_{t2[0:4]}.png", dpi=300
)

In [ ]:
# moving window
mw = 24 * 3  # hours

plt.figure()
ds = ds_all.sel(dataset="CO2")
ds.value.plot(ls="", marker=".")

ds.value.dropna("time").rolling(time=mw, center=True, min_periods=mw / 2).mean().plot(
    ls="-"
)

# check QCflag = 3
ds.where(ds.QCflag == 3).value.plot(ls="", marker=".", c="r")


plt.show()

In [ ]:
# Remove outliers that exceed 10*stdedeviation and 4* the zscore (see )
# remove outliers for each species sperarately
ds_all_rem_out = ds_all.copy(deep=True) #use deepcopy, otherwise it replaces values in ds_all!
for ds in np.unique(ds_all.species):
    print(f"remove outliers for {ds}:")
    ds_all_rem_out.loc[dict(dataset=ds)] = process_data.rem_out(ds_all.sel(dataset=ds), std_fac=10, z_threshold=4)

ds_all = ds_all_rem_out # use the removed outlier data!

In [ ]:
# Define fit paramaters for the curve fitting (NOAA approach)
from utils.ccg_filter import ccg_filter as ccgfilt
from utils.ccg_filter import ccg_dates
from utils import run_curve_fit

# Default values
fit_params_defaults = {'shortterm': 45, #Short term cutoff value in days for smoothing of data
                'longterm': 667, # smoothing in days. Default: 667
                'numpolyterms': 3, # use only 2 for less than 3 years of data, otherwise use 3 (=quadratic fit)
                'sampleinterval': 1 / 24,  # 1h
                'numharmonics': 4}

fit_properties = {}
for dataset in ds_all.dataset:
    dataset_name = dataset.values.item()  # Convert numpy array to a hashable type
    fit_properties[dataset_name] = fit_params_defaults.copy()

### Compare to CAMS
For more CAMS comparison figures, check check_cams.ipynb


In [ ]:
cams_best_grid = xr.open_dataset(f"{data_path}/level3/cams/cams_best_grid_merged_MKN.nc")

In [ ]:
## resample observations
# remove unnecessary variables for simplication
variables_to_keep = ["time", "dataset", "value", "value_unc", "unit", "species"]
ds_all_simple = ds_all.drop_vars(set(ds_all.variables) - set(variables_to_keep))


# Resample only time-dependent variables
time_dependent_variables = [
    var for var in ds_all_simple.data_vars if "time" in ds_all_simple[var].dims
]
non_time_dependent_variables = [
    var
    for var in ds_all_simple.data_vars
    if var not in time_dependent_variables or not "time"
]

ds_all_3h = (
    ds_all_simple[time_dependent_variables].resample(time="3h").mean(keep_attrs=True)
)
ds_all_6h = (
    ds_all_simple[time_dependent_variables].resample(time="6h").mean(keep_attrs=True)
)
# add non-time dependent variables again
for var in non_time_dependent_variables:
    ds_all_3h[var] = ds_all_simple[var]
    ds_all_6h[var] = ds_all_simple[var]

In [ ]:
###### --- FOR ALL comparison FIGURES --- #####
## Select the time period and species to plot


tsel1 = "2020-01-01"
tsel2 = "2023-12-31"

sel_species = ds_all.species
unique_species = np.unique(sel_species)

species_sel = unique_species

# species_sel = ["CO2", "CO"]

In [ ]:
## plot cams versus observations
alpha = 0.7
ms = 8

fig, axs = plt.subplots(
    len(species_sel), 1, sharex=True, figsize=(10, len(species_sel) * 2)
)
for s, ax in zip(species_sel, axs):
    if s == "CH4":
        ds = ds_all_6h  # ch4 has 6h resolution
    else:
        ds = ds_all_3h

    # OR: use hourly observations
    # ds = ds_all

    ds_sel = ds.sel(time=slice(tsel1, tsel2)).where(ds.species == s, drop=True)
    # select cams datasets that exist for this species
    cams_datasets_sel = [
        cams_sel
        for cams_sel in cams_best_grid.dataset.values
        if f"{s.lower()}_" in cams_sel
    ]
    cams = cams_best_grid.sel(dataset=cams_datasets_sel, time=slice(tsel1, tsel2))

    # function to plot each subplot
    def plot_data(ds, label):
        if "flask" in str(ds.dataset.values):
            m = "."
            l = ""
        else:
            m = "."
            l = "-"
        ds.plot(
            ax=ax,
            ls=l,
            marker=m,
            markeredgewidth=0,
            alpha=alpha,
            markersize=ms,
            label=f"{label} {str(ds.dataset.values)}",
        )

    # plot obs
    if len(ds_sel.dataset) > 1:  # several obs datasets with same species
        # plot only if we have data in our given time period:
        variables_to_keep = [
            spec.dataset.values
            for spec in ds_sel.dataset
            if ds_sel.sel(dataset=spec)["value"].isnull().all() == False
        ]
        for ds_i in ds_sel.sel(dataset=variables_to_keep).dataset:
            plot_data(ds_sel.sel(dataset=ds_i.values)["value"], "obs")
    else:
        plot_data(ds_sel.sel(dataset=s)["value"], "obs")

    # plot cams
    if len(cams.dataset) > 1:  # several cams datasets with same species
        # plot only if we have data in our given time period:
        variables_to_keep = [
            spec.dataset.values
            for spec in cams.dataset
            if cams.sel(dataset=spec)["value"].isnull().all() == False
        ]
        for cams_i in cams.sel(dataset=variables_to_keep).dataset:
            plot_data(cams.sel(dataset=cams_i)["value"], "cams")
    else:
        plot_data(cams.isel(dataset=0)["value"], "cams")

    ax.set_ylabel(f"{ds_sel.species[0].values} ({ds_sel.unit[0].values})")
    ax.set_xlabel("")
    ax.set_title("")
    ax.legend()
plt.tight_layout()
plt.show()

### Check Seasonal and daily cycles

In [ ]:
## Remove some outliers, especially wild fire CO

# remove outliers for each species sperarately
ds_all_rem_out = ds_all.copy(deep=True) #use deepcopy, otherwise it replaces values in ds_all!
ds_all_3h_rem_out = ds_all_3h.copy(deep=True)

std_fac = 10
z_threshold =4
for ds in species_sel:
    print(f"remove outliers for {ds}:")
    ds_all_rem_out.loc[dict(dataset=ds)] = process_data.rem_out(ds_all.sel(dataset=ds), std_fac=std_fac, z_threshold=z_threshold)
    ds_all_3h_rem_out.loc[dict(dataset=ds)] = process_data.rem_out(ds_all_3h.sel(dataset=ds),  std_fac=std_fac, z_threshold=z_threshold)

# ## plot removed data (this is already plotted in process_data in the outlier removal)
# var = "value"
# for s in species_sel:
#     f, axs = plt.subplots(2, 1, sharex=True)
#     plt.suptitle("Removed outliers")
#     ds_all.sel(dataset=s)[var].plot(ls="", marker="o", ax=axs[0])
#     ds_all.sel(dataset=s)[var + "_unc"].plot(ls="", marker="o", ax=axs[1])
#     # new data:
#     ds_all_rem_out.sel(dataset=s)[var].plot(ls="", marker=".", ax=axs[0])
#     ds_all_rem_out.sel(dataset=s)[var + "_unc"].plot(ls="", marker=".", ax=axs[1])
#     plt.show()

In [ ]:
## Example for seasonal cycle for Methane with 2-ways of showing uncertainty in 2 different ways
#freq = 'month'
freq = 'dayofyear'
xvals = ds_all.sel(dataset='CH4').groupby(f"time.{freq}").mean()
std1 = ds_all.sel(dataset='CH4').groupby(f"time.{freq}").std()['value'] # standarddeviation of the grouped values of the measurements
std2 = ds_all.sel(dataset='CH4').groupby(f"time.{freq}").mean()["value_unc"] # group-mean of the measurement uncertainties, which is just the hourly standard deviation!
#std4 = 
plt.figure()
xvals["value"].plot()
#std["value"].plot()
plt.fill_between(xvals[freq],xvals["value"],xvals["value"]+std1,alpha=.3,label=f'std. dev. of value in {freq}')
plt.fill_between(xvals[freq],xvals["value"],xvals["value"]-std2,alpha=.3, label='measurement uncertainty')
plt.legend()
plt.show()

In [ ]:
## plot  SEASONAL CYCLE and DIURNAL cams versus observations

alpha = 0.7
ms = 8

# dont use all cams-products
cams_mask = ~np.isin(cams_best_grid["dataset"], "co2_egg4", "ch4_egg4")
cams_use = cams_best_grid.sel(dataset=cams_mask)
# Or: use all:
cams_use = cams_best_grid

for freq in ["dayofyear", "hour"]:
    fig, axs = plt.subplots(
        len(species_sel), 1, sharex=True, figsize=(5, len(species_sel) * 2)
    )
    for s, ax in zip(species_sel, axs):
        if s == "CH4":
            ds = ds_all  # ch4 has 6h resolution
        else:
            ds = ds_all

        ds_sel = ds.sel(time=slice(tsel1, tsel2)).where(ds.species == s, drop=True)
        # select cams datasets that exist for this species
        cams_datasets_sel = [
            cams_sel
            for cams_sel in cams_use.dataset.values
            if f"{s.lower()}_" in cams_sel
        ]
        cams = cams_use.sel(dataset=cams_datasets_sel, time=slice(tsel1, tsel2))

        # function to plot each subplot
        def plot_data(ds, label):
            # other option: use moving-averaged mean:
            mw_temp=30
            #ds = ds.rolling(time=mw_temp, center=True, min_periods=mw_temp / 2).mean()
            ds.groupby(f"time.{freq}").mean().plot(
                ax=ax,
                ls="-",
                marker="",
                markeredgewidth=0,
                alpha=alpha,
                markersize=ms,
                label=f"{label} {str(ds.dataset.values)}",
            )

        # plot obs
        if len(ds_sel.dataset) > 1:  # several obs datasets with same species
            # plot only if we have data in our given time period:
            variables_to_keep = [
                spec.dataset.values
                for spec in ds_sel.dataset
                if ds_sel.sel(dataset=spec)["value"].isnull().all() == False
            ]
            for ds_i in ds_sel.sel(dataset=variables_to_keep).dataset:
                plot_data(ds_sel.sel(dataset=ds_i.values)["value"], "obs")
        else:
            plot_data(ds_sel.sel(dataset=s)["value"], "obs")

        # plot cams
        if len(cams.dataset) > 1:  # several cams datasets with same species
            # plot only if we have data in our given time period:
            variables_to_keep = [
                spec.dataset.values
                for spec in cams.dataset
                if cams.sel(dataset=spec)["value"].isnull().all() == False
            ]
            for cams_i in cams.sel(dataset=variables_to_keep).dataset:
                plot_data(cams.sel(dataset=cams_i)["value"], "cams")
        else:
            plot_data(cams.isel(dataset=0)["value"], "cams")

        ax.set_ylabel(f"{ds_sel.species[0].values} ({ds_sel.unit[0].values})")
        ax.set_xlabel("")
        ax.set_title("")
        ax.legend()

    if freq == "month":
        tit = "Seasonal"
    elif freq == "hour":
        tit = "Diurnal"
    elif freq == "dayofyear":
        tit = "Seasonal (daily data)"

    plt.suptitle(f"{tit} cycle")
    plt.tight_layout()
    plt.show()

## Remove trends

### Use the NOAA method of curve fitting

The method is described: https://gml.noaa.gov/ccgg/mbl/crvfit/crvfit.html 

The code is available at: https://gml.noaa.gov/aftp/user/thoning/ccgcrv/

In [ ]:
from utils.ccg_filter import ccg_filter as ccgfilt
from utils.ccg_filter import ccg_dates

In [ ]:
ds = ds_all.sel(dataset="CO2", time=slice("2020-01-01", "2022-12-31"))["value"]
ds_nonan = ds.where(np.isnan(ds) == False, drop=True)

xp = ds_nonan.time
yp = ds_nonan.values

# get time as decimal date
xp_dec = [ccg_dates.decimalDateFromDatetime(d) for d in pd.to_datetime(xp)]

plt.subplots()
plt.plot(xp, yp, ls="", marker=".")
plt.show()

In [ ]:
# create the ccgfilt object
# Note:  For less than 3 years of data it is best
# to use a linear term for the polynomial part of the function (k=2)

filt = ccgfilt.ccgFilter(
    xp_dec,
    yp,  # 
    shortterm=80, # To decide smoothing! #cutoff frequency (days) for smoothing of data (default=80days)
    #longterm=667, #cutoff for extracting trend (default = 667 days)
    numpolyterms=2,  # number of polynomial terms, 3 = quadratic
    # numharmonics, timezero, gap, debug
    sampleinterval=1/24,  # Interval in days between samples, calculate equally spaced values at this interva
    numharmonics=4,
    #debug=True
)

In [ ]:
# get x,y data for plotting
#other option: use x0 = filt.xinterp (interpolated x instead of original xp)
y1 = filt.getFunctionValue(xp_dec)
y2 = filt.getPolyValue(xp_dec)
y3 = filt.getSmoothValue(xp_dec)
y4 = filt.getTrendValue(xp_dec)

plt.figure()
plt.plot(xp_dec, yp, ls="", marker=".", label="obs")
plt.plot(xp_dec, y1, label="Function fit")
plt.plot(xp_dec, y2, label="Polynomial")
plt.plot(xp_dec, y3, label="smoothed data")  # = function fit + filtered residuals
plt.plot(xp_dec, y4, label="trend")  # trend with seasonal cycle removed
plt.legend()
# plt.xlim([2000, 2005])
#plt.ylim([406, 420])

In [ ]:
# Detrended seasonal Cycle
# x and y are original data points
trend = filt.getTrendValue(xp_dec)
detrend = yp - trend #data minus trend

x0 = filt.xinterp
harmonics = filt.getHarmonicValue(x0)
smooth_cycle = harmonics + filt.smooth - filt.trend #smoothed curve (plus seasonal harmonics) minus trend

plt.subplots()
plt.plot(xp_dec,detrend,ls='',marker='.') #
plt.plot(x0, smooth_cycle)
plt.title('Detrended data and smoothed fit')

In [ ]:
## Make a dataframe of all the outputs
df = pd.DataFrame(
    {
        "time": xp,
        "fit_vals": filt.getFunctionValue(xp_dec),
        "smoothed_vals": filt.getSmoothValue(xp_dec),
        "poly": filt.getPolyValue(xp_dec),
        "harmonics_vals": filt.getHarmonicValue(xp_dec),
        "trend_vals": filt.getTrendValue(xp_dec)
    }
)
df.set_index("time", inplace=True)
print(df.head())


## dataframe with direct values (on interpolated grid)
x0 = filt.xinterp
datetimes = [ccg_dates.datetimeFromDecimalDate(d) for d in x0]
df_interp = pd.DataFrame(
    {
        "time": datetimes,
        "smooth": filt.smooth,
        "trend": filt.trend,
        "smooth_vals": filt.getSmoothValue(x0),
        "poly_vals": filt.getPolyValue(x0),
        "trend_vals": filt.getTrendValue(x0),
        "harmonics_vals": filt.getHarmonicValue(x0),
        "seasonal_detrend": filt.getHarmonicValue(x0) + filt.smooth - filt.trend,
    }
)
df_interp.set_index("time", inplace=True)
df_interp.head()

# => the second one uses interpolated values for the missing dates, but otherwise the both dataframes are basically the same
# => don't really need to save both, maybe just use the interpolated one!

# smooth_cycle = harmonics + filt.smooth - filt.trend #smoothed curve (plus seasonal harmonics) minus trend

In [ ]:
## plot seasonal cycle of detrended data
ref = ds.mean() # substract mean value to obtain positive/neg. seasonality
fig, ax = plt.subplots()
(ds_nonan-ref).groupby("time.month").mean().plot(label="normal")
df_interp['seasonal_detrend'].to_xarray().groupby("time.month").mean().plot(ax=ax, label="detrended seasonal cycle")
plt.legend()

In [ ]:
fig, ax = plt.subplots()

ds.plot(ax=ax, marker=".", ls="")  #

ax.plot(xp,df['smoothed_vals'],label='curve fit',c='lightblue')


ax.plot(df_interp.index,df_interp['seasonal_detrend']+df_interp['smooth_vals'].mean(), label='detrended fit') ## Add mean value to detrended to obtain same magnitude
plt.title('Smoothed and detrended fit')
plt.legend()

#### Do the same NOAA fit but for all species

In [ ]:
## function to select correct data and create the fit/filter object
def run_ccgfilter(
    dataset_str,
    ds,
    t1="2020-01-01",
    t2="2022-12-31",
    shortterm=80,
    longterm=667,
    numpolyterms=2,
    sampleinterval=1 / 24,  #if not given, determine from the data
    numharmonics=4,
):
    ## prepare data
    ds_sel = ds.sel(time=slice(t1, t2))
    ds_nonan = ds_sel.where(np.isnan(ds_sel) == False, drop=True)

    xp = ds_nonan.time
    yp = ds_nonan.values

    if len(xp)>0 and len(yp)>0: #if we have data
        # get time as decimal date
        xp_dec = [ccg_dates.decimalDateFromDatetime(d) for d in pd.to_datetime(xp)]

        # create the ccgfilt object
        # Note:  For less than 3 years of data it is best
        # to use a linear term for the polynomial part of the function (k=2)
        filt = ccgfilt.ccgFilter(
            xp_dec,
            yp,  #
            shortterm=shortterm,  # To decide smoothing! #cutoff frequency (days) for smoothing of data (default=80days)
            longterm=longterm, #cutoff for extracting trend (default = 667 days)
            numpolyterms=numpolyterms,  # number of polynomial terms, 3 = quadratic. For <3 years, use 2
            sampleinterval=sampleinterval,  # Interval in days between samples, calculate equally spaced values at this interval
            numharmonics=numharmonics,
            # debug=True
        )
    else:
        filt = None
    return filt

In [ ]:
# run curve fitting for each species/dataset and save in a seperate dataset

#sel_species = ds_all.species
#unique_species = np.unique(sel_species)
#data_to_fit = unique_species
data_to_fit = selected_data

# select time period
t1 = "2020-01-01"
t2 = "2023-12-31"
fit_shortterm = 80

datasets = []
for dataset_str in data_to_fit:
    ## prepare data and nooa-curve fit run (use the interpolated data)
    # slect dataset
    ds_sel = ds_all.sel(dataset=dataset_str)["value"]
    #run curve fitting
    filt = run_ccgfilter(
        dataset_str,
        ds_sel,
        t1=t1,
        t2=t2,
        shortterm=fit_shortterm, #default: 80
        longterm=667, #default: 667
        numpolyterms=2, #default: 3 (quadratic). For <3 years, use 2 (see conclusions on https://gml.noaa.gov/ccgg/mbl/crvfit/crvfit.html)
        sampleinterval=1 / 24,
        numharmonics=4,
    )

    # dataframe with fit values (on interpolated grid)
    if filt is not None:
        x0 = filt.xinterp
        datetimes = [ccg_dates.datetimeFromDecimalDate(d) for d in x0]
        df_interp = pd.DataFrame(
            {
                "time": datetimes,
                "smooth": filt.smooth,
                "trend": filt.trend,
                "smoothed_vals": filt.getSmoothValue(x0),
                "trend_vals": filt.getTrendValue(x0),
                "harmonic_vals": filt.getHarmonicValue(x0),
                "seasonal_detrend": filt.getHarmonicValue(x0) + filt.smooth - filt.trend,
            }
        )
        df_interp.set_index("time", inplace=True)
        ds_interp = df_interp.to_xarray()
        ds_interp = ds_interp.assign_coords(dataset=dataset_str)
        datasets.append(ds_interp)
    else:
        continue

# save all in one xarray dataset
ds_fit_all = xr.concat(datasets, dim="dataset")
ds_fit_all

In [ ]:
### Plot all species with curve fit

# species to plot (all available, or make a selection)
sel_species = ds_all.species
unique_species = np.unique(sel_species)
skip_flask = True  # don't use flask datasets

unique_species = ds_fit_all.dataset

# dataset to use:
# ds_data = ds_all #normal measurement data
ds_data = ds_all_rem_out  # use data with removed outliers!!



# function to plot each subplot
# time series plot
def plot_data(ds_temp, ds_temp_fit, s, ax=None, **kwargs):
    if ax is None:
        ax = plt.gca()
    ds_temp.plot(
        ax=ax,
        ls="",
        marker=".",
        alpha=0.7,
        label=str(s),
        markeredgewidth=0,
    )
    ds_temp_fit["smoothed_vals"].plot(ax=ax, label="curve fit", c="lightblue")
    ax.plot(
        ds_temp_fit.time,
        ds_temp_fit["seasonal_detrend"] + ds_temp_fit["smoothed_vals"].mean(),
        label="detrended fit",
    )  ## Add mean value to detrended to obtain same magnitude


# seasonality plot
def plot_cycle(ds_temp, ds_temp_fit, s, freq="month", ax=None, **kwargs):
    if ax is None:
        ax = plt.gca()

    ref = (
        ds_temp.mean()
    )  # use mean value from measurements to obtain positive/neg. seasonality (ds_temp-ref) or absolute values (ds_temp_fit +ref)
    ds_temp.groupby(f"time.{freq}").mean().plot(ax=ax, label="normal", c="lightblue")
    (ds_temp_fit["seasonal_detrend"] + ref).groupby(f"time.{freq}").mean().plot(
        ax=ax, label="detrended seasonal cycle", c="C1"
    )


## Start figure
fig = plt.figure(figsize=(8, len(unique_species) * 2), layout="constrained")
grid = plt.GridSpec(len(unique_species), 4, figure=fig)

# loop through species
for i, s in enumerate(unique_species):
    # select data
    ds_sel = ds_data.where(ds_data.species == s, drop=True).sel(time=slice(t1, t2))
    ds_sel_fit = ds_fit_all.where(
        ds_fit_all.dataset == s, drop=True
    )  # I applied the fit only to 2020 to 2023 and not to flask data

    print(f"plot {s}")

    ## Time series figure
    ax_ts = fig.add_subplot(grid[i, :3])
    ## Seasonal cycle figure
    ax_seas = fig.add_subplot(grid[i, 3], sharey=ax_ts)

    if len(ds_sel.dataset) > 1:  # several datasets with same species (e.g. flask data)
        for ii in ds_sel.dataset:
            # plot data
            # check flask data
            if skip_flask and ("flask" in ii.item()):
                print(f"skip {ii.item()}")
                continue
            else:
                plot_data(
                    ds_sel.sel(dataset=ii).value,
                    ds_sel_fit.sel(
                        dataset=s
                    ),  # no flask data (that would require to use ii instead of s)
                    ii.item(),
                    ax_ts,
                )
                # plot seasonal cycle
                plot_cycle(
                    ds_sel.sel(dataset=ii).value,
                    ds_sel_fit.sel(dataset=s),
                    ii.item(),
                    freq="month",
                    ax=ax_seas,
                )
    else:
        plot_data(ds_sel.sel(dataset=s).value, ds_sel_fit.sel(dataset=s), s, ax_ts)
        # plot seasonal cycle
        plot_cycle(
            ds_sel.sel(dataset=s).value,
            ds_sel_fit.sel(dataset=s),
            s,
            freq="month",
            ax=ax_seas,
        )

    # axes properties time series
    ax_ts.set_ylabel(f"{ds_sel.species.values[0]} ({ds_sel.unit.values[0]})")
    ax_ts.set_xlabel("")
    ax_ts.set_xlim(np.array([t1, t2], dtype="datetime64"))
    ax_ts.spines["top"].set_visible(False)
    ax_ts.spines["right"].set_visible(False)
    ax_ts.set_title("")

    if i < len(unique_species) - 1:  # all except lowest axis
        plt.setp(
            ax_ts.get_xticklabels(), visible=False
        )  # remove xticklabels except for lowest plot

    # axes properties seasonal cycle
    # ax_seas.set_yticklabels('') #this removes ylabels for both axes! Therefore better use:
    ax_seas.tick_params(labelleft=False)
    ax_seas.set_title("")
    ax_seas.spines["top"].set_visible(False)
    ax_seas.spines["right"].set_visible(False)
    months = range(1, 13, 3)
    ax_seas.set_xticks(
        months,
        [dt.datetime.strptime(str(month), "%m").strftime("%b") for month in months],
    )
    # ax_seas.grid(color='white',zorder=4,axis='x')
    # ax_seas.set_axisbelow(False)

    # different for lowest plot
    if i < len(unique_species) - 1:  # all except lowest axis
        plt.setp(
            ax_seas.get_xticklabels(), visible=False
        )  # remove xticklabels except for lowest plot
        ax_seas.spines["bottom"].set_visible(False)
        ax_seas.set_xticks([])
        ax_seas.set_xticklabels("")
        # ax_seas.set_xticks()
        ax_seas.set_xlabel("")


handles, labels = ax_ts.get_legend_handles_labels()
# manually adapt legend:
labels[0] = "1h data"
plt.subplot(grid[0, :3]).legend(handles, labels, loc="lower right")
fig.align_ylabels()


# place title on first axis
plt.subplot(grid[0, :3]).set_title("Mt. Kenya measurements and curve fit", loc="left")
plt.subplot(grid[0, 3]).set_title("Seasonal cycle", loc="left")

#plt.savefig(f"{dir_save}timeseries_and_seasonal_{t1[:4]}_{t2[:4]}.png", dpi=300)

In [ ]:
### Same but with normalized seasonal cycle

# species to plot (all available, or make a selection)
sel_species = ds_all.species
unique_species = np.unique(sel_species)
skip_flask = True #don't use flask datasets

#dataset to use: 
#ds_data = ds_all #normal measurement data
ds_data = ds_all_rem_out #use data with removed outliers!!


# select time period
t1 = "2020-01-01"
t2 = "2023-12-31"

# seasonality plot
def plot_cycle(ds_temp, ds_temp_fit, s, freq="month", ax=None, **kwargs):
    if ax is None:
        ax = plt.gca()

    ref = (
        ds_temp.mean()
    )  # use mean value from measurements to obtain positive/neg. seasonality (ds_temp-ref) or absolute values (ds_temp_fit +ref)
    (ds_temp-ref).groupby(f"time.{freq}").mean().plot(ax=ax, label="normal", c="lightblue")
    (ds_temp_fit["seasonal_detrend"]).groupby(f"time.{freq}").mean().plot(
        ax=ax, label="detrended seasonal cycle", c="C1"
    )

## Start figure
fig = plt.figure(figsize=(8, len(unique_species) * 2),
    layout="constrained")
grid = plt.GridSpec(len(unique_species), 4,figure=fig)

#loop through species
for i,s in enumerate(unique_species):
    #select data
    ds_sel = ds_data.where(ds_data.species == s, drop=True).sel(time=slice(t1, t2))
    ds_sel_fit = ds_fit_all.where(ds_fit_all.dataset == s, drop=True) # I applied the fit only to 2020 to 2023 and not to flask data

    print(f'plot {s}')

    ## Time series figure
    ax_ts = fig.add_subplot(grid[i, :3])
    ## Seasonal cycle figure
    ax_seas = fig.add_subplot(grid[i, 3]) #dont share yaxis #,sharey=ax_ts

    if len(ds_sel.dataset) > 1:  # several datasets with same species (e.g. flask data)
        for ii in ds_sel.dataset:
            # plot data
            # check flask data
            if skip_flask and ("flask" in ii.item()):
                print(f'skip {ii.item()}')
                continue
            else:
                plot_data(
                    ds_sel.sel(dataset=ii).value,
                    ds_sel_fit.sel(dataset=s), #no flask data (that would require to use ii instead of s)
                    ii.item(),
                    ax_ts,
                )
                #plot seasonal cycle
                plot_cycle(
                    ds_sel.sel(dataset=ii).value,
                    ds_sel_fit.sel(dataset=s),
                    ii.item(), 
                    freq='month',
                    ax=ax_seas)
    else:
        plot_data(ds_sel.sel(dataset=s).value,ds_sel_fit.sel(dataset=s), s, ax_ts)
        #plot seasonal cycle
        plot_cycle(ds_sel.sel(dataset=s).value,ds_sel_fit.sel(dataset=s),s, freq='month', ax=ax_seas)

    # axes properties time series
    ax_ts.set_ylabel(f"{ds_sel.species.values[0]} ({ds_sel.unit.values[0]})")
    ax_ts.set_xlabel("")
    ax_ts.set_xlim(np.array([t1, t2], dtype="datetime64"))
    ax_ts.spines['top'].set_visible(False)
    ax_ts.spines['right'].set_visible(False)
    ax_ts.set_title("")
    
    if i<len(unique_species)-1: #all except lowest axis
        plt.setp(ax_ts.get_xticklabels(), visible=False) #remove xticklabels except for lowest plot

    #axes properties seasonal cycle
    #ax_seas.set_yticklabels('') #this removes ylabels for both axes! Therefore better use:
    #ax_seas.tick_params(labelleft=False)
    ax_seas.set_ylabel('')
    ax_seas.set_title('')
    ax_seas.spines['top'].set_visible(False)
    ax_seas.spines['right'].set_visible(False)
    months = range(1,13,3)
    ax_seas.set_xticks(months, [dt.datetime.strptime(str(month), '%m').strftime("%b") for month in months])
    
    #different for lowest plot
    if i<len(unique_species)-1: #all except lowest axis
        plt.setp(ax_seas.get_xticklabels(), visible=False) #remove xticklabels except for lowest plot
        ax_seas.spines['bottom'].set_visible(False)
        ax_seas.set_xticks([])
        ax_seas.set_xticklabels('')
        #ax_seas.set_xticks()
        ax_seas.set_xlabel('')



handles, labels = ax_ts.get_legend_handles_labels()
#manually adapt legend: 
labels[0] = '1h data'
plt.subplot(grid[0, :3]).legend(handles, labels, loc="lower right")
fig.align_ylabels()


#place title on first axis
plt.subplot(grid[0, :3]).set_title("Mt. Kenya measurements and curve fit",loc='left')
plt.subplot(grid[0,3]).set_title("Seasonal cycle",loc='left')

plt.savefig(
    f"{dir_save}timeseries_and_seasonal_2020_2023_2.png", dpi=300
)

## Other methods

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

ds = ds_all.sel(dataset="CO2")["value"]
x = ds[~np.isnan(ds)]
# x = ds.time.value[~np.isnan(ds['value'].values)]
# y = ds['value'].values[~np.isnan(ds['value'].values)]

decomposition = seasonal_decompose(x, model="additive", period=365 * 24)


ts_freq = 365 * 24
decomposition = seasonal_decompose(
    x,
    model="additive",
    period=ts_freq,  # extrapolate_trend=ts_freq
)  # Assuming hourly data
plt.figure()
decomposition.plot()
plt.show()

detrended_data = x - decomposition.trend  # remove trend

# plt.figure()
# detrended_data.plot()
# plt.show()


detrended_data += (
    decomposition.seasonal + decomposition.observed.mean()
)  # add the mean value as intercept and seasonality

fig, ax = plt.subplots()
x.plot(ax=ax, label="normal")
detrended_data.plot(ax=ax, label="detrended", ls=":")
plt.legend()
plt.show()

In [ ]:
## Other method: STL -> Season-Trend decomposition using LOESS (takes long time with daily data!)
from statsmodels.tsa.seasonal import STL


ds = ds_all.sel(dataset="CO2").drop("dataset")["value"]


x = ds[~np.isnan(ds)].to_dataframe()


stl = STL(x, period=24 * 365)


res = stl.fit()  # this takes 4min!


fig = res.plot()

In [ ]:
# show detrended data
detrended_data = x.value - res.trend  # remove trend

detrended_data += (
    # res2.seasonal
    +res.observed.mean().values
)  # add the mean value as intercept and seasonality (?)

ds_sel = ds_all.sel(dataset="CO2").drop("dataset")["value"]

# plot detrended data and the seasonal cycle
fig, axs = plt.subplots(2, 1, layout="constrained")
ax = axs[0]
ds_sel.plot(ax=ax, ls="", marker=".", label="normal")
detrended_data.plot(ax=ax, label="detrended", ls=":", marker=".")
ax.legend()

## Show seasonal cycle (detended and not)
ax = axs[1]
ds_sel.groupby("time.month").mean().plot(ax=ax, label="normal seasonal cycle")
detrended_data.to_xarray().groupby("time.month").mean().plot(
    ax=ax, label="detrended seasonal cycle"
)
ax.legend()
plt.show()

In [ ]:
## do the same but with monthly means
from statsmodels.tsa.seasonal import STL

dsM = ds_all.sel(dataset="CO2").resample(time="MS").mean().drop("dataset")["value"]

x2 = dsM[~np.isnan(dsM)].to_dataframe()

# fig, ax = plt.subplots()
stl2 = STL(x2, period=12)
res2 = stl2.fit()
res2.plot()

In [ ]:
detrended_data = x2.value - res2.trend  # remove trend

# plt.figure()
# detrended_data.plot()
# plt.show()


detrended_data += (
    # res2.seasonal
    +res2.observed.mean().values
)  # add the mean value as intercept and seasonality (?)

fig, ax = plt.subplots()
dsM.plot(ax=ax, label="normal")
detrended_data.plot(ax=ax, label="detrended", ls=":", marker=".")
plt.legend()
plt.show()

In [ ]:
## Show seasonal cycle (detended and not)
plt.figure()


dsM.groupby("time.month").mean().plot(label="normal")


detrended_data.to_xarray().groupby("time.month").mean().plot(label="detrend")
plt.legend()


plt.show()

In [ ]:
ds = ds_all.sel(dataset="CO2")["value"]
ds[~np.isnan(ds)].to_dataframe()

In [ ]:
## time series difference
# => this removes also the seasonality!
ds = ds_all.sel(dataset="CO2")["value"]
diff = ds.diff(dim="time")
ds_mean = ds.mean(dim="time")
plt.figure()
ds.plot(label="simpel detrended")
(diff + ds_mean).plot(ls=":", label="detrended (diff)")
plt.show()

### Other methods

In [ ]:
##not working...?
# import statsmodels.api as sm

# Y = x.values
# X = x.time.values
# X = sm.add_constant(X)
# model = sm.OLS(Y, X)
# results = model.fit()

In [ ]:
## SARIMA

import statsmodels.api as sm

mod = sm.tsa.statespace.SARIMAX(x.values, trend="c", order=(1, 1, 1))

res = mod.fit(disp=False)
print(res.summary())

In [ ]:
## Check autocorrelation
data = x.values
data["ln"] = np.log(data)
data["D.ln"] = data["ln"].diff()

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

fig = sm.graphics.tsa.plot_acf(data.iloc[1:]["D.ln"], lags=40, ax=axes[0])
fig = sm.graphics.tsa.plot_pacf(data.iloc[1:]["D.ln"], lags=40, ax=axes[1])